In [1]:
!pip install -q kagglehub
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
path = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")
print("Dataset downloaded to:", path)

import os
os.listdir(path)

Using Colab cache for faster access to the 'diseases-and-symptoms-dataset' dataset.
Dataset downloaded to: /kaggle/input/diseases-and-symptoms-dataset


['Final_Augmented_dataset_Diseases_and_Symptoms.csv']

In [3]:
import pandas as pd

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
print("Files found:", csv_files)

df = pd.read_csv(os.path.join(path, csv_files[0]))
print(df.shape)
print(df.columns.tolist()[:10])  # confirm which column is the disease label
df.head()

Files found: ['Final_Augmented_dataset_Diseases_and_Symptoms.csv']
(246945, 378)
['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms', 'sharp chest pain', 'dizziness', 'insomnia', 'abnormal involuntary movements', 'chest tightness']


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [4]:
def normalize_symptom(value):
    import re
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s

label_col = df.columns[0]  # first column is the disease label in this dataset
SYMPTOM_COLS = [c for c in df.columns if c != label_col]

print("Label column:", label_col)
print(f"{len(SYMPTOM_COLS)} symptom columns")

df[label_col] = df[label_col].astype(str).str.strip()

symptom_vocab = [normalize_symptom(c) for c in SYMPTOM_COLS]
df = df.rename(columns=dict(zip(SYMPTOM_COLS, symptom_vocab)))

Label column: diseases
377 symptom columns


In [5]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

X = df[symptom_vocab].to_numpy(dtype=np.float32)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[label_col])
label_classes = list(label_encoder.classes_)

print(X.shape, y.shape, f"{len(label_classes)} classes")

(246945, 377) (246945,) 773 classes


In [6]:

from sklearn.model_selection import train_test_split

MIN_SAMPLES_PER_CLASS = 5

class_counts = pd.Series(y).value_counts()
keep_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index
keep_mask = pd.Series(y).isin(keep_classes).to_numpy()

X_filtered = X[keep_mask]
y_filtered = y[keep_mask]


dropped = len(y) - len(y_filtered)
print(f"Dropped {dropped} rows across {class_counts.size - len(keep_classes)} rare-disease classes")
print(f"Remaining: {X_filtered.shape[0]} rows, {len(keep_classes)} classes")

X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)



Dropped 122 rows across 52 rare-disease classes
Remaining: 246823 rows, 721 classes


In [7]:
from scipy.sparse import csr_matrix
from sklearn.ensemble import RandomForestClassifier

X_train_sparse = csr_matrix(X_train)
X_test_sparse = csr_matrix(X_test)

clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    max_samples=0.5,
    n_jobs=2,
    random_state=42,
)
clf.fit(X_train_sparse, y_train)
print("Trained.")

Trained.


In [8]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = clf.predict(X_test_sparse)
v2_accuracy = accuracy_score(y_test, y_pred)
v1_accuracy = 0.8162  # replace with your actual v1 number
print(f"v2 (Random Forest) accuracy: {v2_accuracy:.4f}")
print(f"v1 (Decision Tree) accuracy: {v1_accuracy:.4f}")
print(f"Difference: {v2_accuracy - v1_accuracy:+.4f}")
print()

present_labels = sorted(set(y_test) | set(y_pred))
present_names = [label_encoder.classes_[i] for i in present_labels]
print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

importances = pd.Series(clf.feature_importances_, index=symptom_vocab).sort_values(ascending=False)
print("Top 20 most important symptoms:")
print(importances.head(20))

v2 (Random Forest) accuracy: 0.6588
v1 (Decision Tree) accuracy: 0.8162
Difference: -0.1574

                                                          precision    recall  f1-score   support

                               abdominal aortic aneurysm       0.00      0.00      0.00        28
                                        abdominal hernia       1.00      0.73      0.84        81
                                         abscess of nose       1.00      0.57      0.73        58
                                     abscess of the lung       0.00      0.00      0.00         4
                                  abscess of the pharynx       0.00      0.00      0.00        68
                                    acanthosis nigricans       0.00      0.00      0.00         6
                                               acariasis       0.00      0.00      0.00         7
                                               achalasia       0.00      0.00      0.00        17
                        

In [9]:
import json
import joblib

os.makedirs("model_artifacts", exist_ok=True)

joblib.dump(clf, "model_artifacts/v2_random_forest.joblib")

label_classes = [label_encoder.classes_[i] for i in clf.classes_]
with open("model_artifacts/label_classes.json", "w") as f:
    json.dump(label_classes, f, indent=2)

with open("model_artifacts/symptom_vocab.json", "w") as f:
    json.dump(symptom_vocab, f, indent=2)

assert len(label_classes) == len(clf.classes_)
assert clf.predict_proba(X_test_sparse[:1]).shape[1] == len(label_classes), "predict_proba width mismatch — do not ship"

print(f"{len(label_classes)} disease classes exported")
!ls -la model_artifacts

import shutil
shutil.make_archive("v2_model_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v2_model_artifacts.zip")

721 disease classes exported
total 503856
drwxr-xr-x 2 root root      4096 Aug 12 16:39 .
drwxr-xr-x 1 root root      4096 Aug 12 16:39 ..
-rw-r--r-- 1 root root     17367 Aug 12 16:39 label_classes.json
-rw-r--r-- 1 root root      8606 Aug 12 16:39 symptom_vocab.json
-rw-r--r-- 1 root root 515902961 Aug 12 16:39 v2_random_forest.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>